INICIALIZAR SPARK

# CELDA 1 — VERIFICACIÓN DEL ENTORNO

In [1]:
import subprocess, sys, os

print("=== VERIFICACIÓN DEL ENTORNO ===")
print(f"Python      : {sys.version.split()[0]}")
print(f"JAVA_HOME   : {os.environ.get('JAVA_HOME', 'NO DEFINIDO')}")

java_ver = subprocess.run(
    ["java", "-version"],
    capture_output=True, text=True
)
print(f"Java version: {java_ver.stderr.splitlines()[0] if java_ver.stderr else 'No encontrado'}")

try:
    import pyspark
    print(f"PySpark     : {pyspark.__version__}")
except ImportError:
    print("PySpark     : NO INSTALADO — revisa requirements.txt o Dockerfile")

=== VERIFICACIÓN DEL ENTORNO ===
Python      : 3.11.15
JAVA_HOME   : /opt/conda
Java version: openjdk version "17.0.18-internal" 2026-01-20
PySpark     : 3.5.1


# CELDA 2 — IMPORTACIONES

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lower, trim, length, split, count,
    desc, regexp_replace, when, array_contains
)

print("Importaciones OK")

Importaciones OK


# CELDA 3 — SPARKSESSION

In [3]:
spark = SparkSession.builder \
    .appName("ETL_Biblia_NTV") \
    .master("local[*]") \
    .config("spark.ui.port", "4040") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("=" * 55)
print(f"  SparkSession lista")
print(f"  Versión Spark : {spark.version}")
print(f"  Master        : {spark.sparkContext.master}")
print(f"  Spark UI      : http://localhost:4040")
print("=" * 55)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 04:20:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


  SparkSession lista
  Versión Spark : 3.5.1
  Master        : local[*]
  Spark UI      : http://localhost:4040


# CELDA 4 — RUTAS

In [4]:
RUTA_BASE    = "/opt/notebooks/proyecto"
RUTA_CSV     = f"{RUTA_BASE}/data/biblia_ntv_.csv"     # nombre exacto del CSV
RUTA_PARQUET = f"{RUTA_BASE}/data/biblia_ntv_procesada.parquet"

In [5]:
# Verificar que el CSV existe antes de continuar
import os
if os.path.exists(RUTA_CSV):
    tam = os.path.getsize(RUTA_CSV) / 1024
    print(f"  CSV encontrado : {RUTA_CSV}")
    print(f"  Tamaño         : {tam:.1f} KB")
else:
    print(f"  ARCHIVO NO ENCONTRADO: {RUTA_CSV}")
    print("  Verifica que el CSV esté en notebooks/proyecto/data/")

  CSV encontrado : /opt/notebooks/proyecto/data/biblia_ntv_.csv
  Tamaño         : 5017.5 KB


# CELDA 5 — EXTRACT

In [6]:
df_raw = spark.read \
    .option("header",      "true") \
    .option("inferSchema", "true") \
    .option("encoding",    "UTF-8") \
    .option("multiLine",   "false") \
    .option("sep",         ",") \
    .csv(RUTA_CSV)

In [7]:
print("=== [EXTRACT] ESQUEMA INFERIDO ===")
df_raw.printSchema()

=== [EXTRACT] ESQUEMA INFERIDO ===
root
 |-- _c0: integer (nullable = true)
 |-- libro: string (nullable = true)
 |-- capitulo: integer (nullable = true)
 |-- verso: integer (nullable = true)
 |-- texto: string (nullable = true)



In [8]:
print("\n=== [EXTRACT] PRIMERAS 5 FILAS ===")
df_raw.show(5, truncate=80)


=== [EXTRACT] PRIMERAS 5 FILAS ===
+---+-------+--------+-----+--------------------------------------------------------------------------------+
|_c0|  libro|capitulo|verso|                                                                           texto|
+---+-------+--------+-----+--------------------------------------------------------------------------------+
|  0|Génesis|       1|    1|   1 El relato de la creación En el principio, Dios creó los cielos y la tierra.|
|  1|Génesis|       1|    2|2 La tierra no tenía forma y estaba vacía, y la oscuridad cubría las aguas pr...|
|  2|Génesis|       1|    3|                               3 Entonces Dios dijo: «Que haya luz»; y hubo luz.|
|  3|Génesis|       1|    4|         4 Y Dios vio que la luz era buena. Luego separó la luz de la oscuridad.|
|  4|Génesis|       1|    5|5 Dios llamó a la luz «día» y a la oscuridad «noche». Y pasó la tarde y llegó...|
+---+-------+--------+-----+--------------------------------------------------------

26/03/26 04:22:36 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , libro, capitulo, verso, texto
 Schema: _c0, libro, capitulo, verso, texto
Expected: _c0 but found: 
CSV file: file:///opt/notebooks/proyecto/data/biblia_ntv_.csv


In [9]:
total_registros = df_raw.count()
print(f"\n  Total registros cargados: {total_registros:,}")


  Total registros cargados: 31,100


# CELDA 6 — INSPECCIÓN DEL CSV

In [10]:
print("=== NOMBRES REALES DE COLUMNAS ===")
print(df_raw.columns)

=== NOMBRES REALES DE COLUMNAS ===
['_c0', 'libro', 'capitulo', 'verso', 'texto']


In [11]:
print("\n=== TIPOS DE DATOS ===")
print(df_raw.dtypes)


=== TIPOS DE DATOS ===
[('_c0', 'int'), ('libro', 'string'), ('capitulo', 'int'), ('verso', 'int'), ('texto', 'string')]


In [12]:
print("\n=== CONTEO POR COLUMNA (nulos) ===")
df_raw.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
]).show()


=== CONTEO POR COLUMNA (nulos) ===


26/03/26 04:22:46 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , libro, capitulo, verso, texto
 Schema: _c0, libro, capitulo, verso, texto
Expected: _c0 but found: 
CSV file: file:///opt/notebooks/proyecto/data/biblia_ntv_.csv


+---+-----+--------+-----+-----+
|_c0|libro|capitulo|verso|texto|
+---+-----+--------+-----+-----+
|  0|    0|       0|    0|    0|
+---+-----+--------+-----+-----+



# CELDA 7 — NORMALIZACIÓN DE NOMBRES DE COLUMNAS

In [13]:
MAPA_COLUMNAS = {
    "libro":     "libro",
    "capitulo":  "capitulo",
    "verso":     "verso",      # ← así viene en tu CSV real
    "versiculo": "verso",      # ← por si acaso
    "verse":     "verso",
    "texto":     "texto",
    "text":      "texto",
}

In [14]:
# Renombrar solo las columnas que existan en el CSV
columnas_actuales = df_raw.columns
df_renombrado = df_raw
for nombre_csv, nombre_std in MAPA_COLUMNAS.items():
    if nombre_csv in columnas_actuales and nombre_csv != nombre_std:
        df_renombrado = df_renombrado.withColumnRenamed(nombre_csv, nombre_std)
        print(f"  Renombrado: '{nombre_csv}' → '{nombre_std}'")

print(f"\n  Columnas después de normalización: {df_renombrado.columns}")


  Columnas después de normalización: ['_c0', 'libro', 'capitulo', 'verso', 'texto']


# CELDA 8 — TRANSFORM

In [15]:
# ── T1: Eliminar nulos ────────────────────────────────────────────────────────
df_sin_nulos = df_renombrado.dropna(
    subset=["libro", "capitulo", "verso", "texto"]
)
eliminados = total_registros - df_sin_nulos.count()
print(f"[T1] Registros eliminados por nulos : {eliminados}")
print(f"[T1] Registros restantes            : {df_sin_nulos.count():,}")

[T1] Registros eliminados por nulos : 0
[T1] Registros restantes            : 31,100


In [16]:
# ── T2: Normalización del texto ───────────────────────────────────────────────
# trim → quitar espacios extremos
# regexp_replace → eliminar puntuación (conservar letras españolas y espacios)
# lower → convertir a minúsculas
df_normalizado = df_sin_nulos.withColumn(
    "texto_limpio",
    lower(
        regexp_replace(
            trim(col("texto")),
            r"[^a-záéíóúüñA-ZÁÉÍÓÚÜÑ\s]",
            ""
        )
    )
)
print("\n[T2] Muestra texto original vs texto_limpio:")
df_normalizado.select("texto", "texto_limpio").show(3, truncate=90)


[T2] Muestra texto original vs texto_limpio:
+------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------+
|                                                                                     texto|                                                                              texto_limpio|
+------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------+
|             1 El relato de la creación En el principio, Dios creó los cielos y la tierra.|                 el relato de la creación en el principio dios creó los cielos y la tierra|
|2 La tierra no tenía forma y estaba vacía, y la oscuridad cubría las aguas profundas; y...| la tierra no tenía forma y estaba vacía y la oscuridad cubría las aguas profundas y el...|
|                                 

In [17]:
# ── T3: Longitud del texto ────────────────────────────────────────────────────
df_con_longitud = df_normalizado.withColumn(
    "longitud_texto",
    length(col("texto_limpio"))
)
print("[T3] Estadísticas de longitud:")
df_con_longitud.select("longitud_texto").describe().show()


[T3] Estadísticas de longitud:
+-------+-----------------+
|summary|   longitud_texto|
+-------+-----------------+
|  count|            31100|
|   mean|132.2525723472669|
| stddev|61.46757054335992|
|    min|                0|
|    max|              652|
+-------+-----------------+



In [18]:
# ── T4: Tokenización ─────────────────────────────────────────────────────────
# split por uno o más espacios → array de palabras por versículo
df_tokenizado = df_con_longitud.withColumn(
    "tokens",
    split(col("texto_limpio"), r"\s+")
).withColumn(
    "num_palabras",
    F.size(col("tokens"))
)

print("[T4] Schema final del DataFrame transformado:")
df_tokenizado.printSchema()

print("\n[T4] Muestra del dataset transformado:")
df_tokenizado.select(
    "libro", "capitulo", "verso",
    "texto_limpio", "longitud_texto", "num_palabras"
).show(5, truncate=70)

df_transformado = df_tokenizado

[T4] Schema final del DataFrame transformado:
root
 |-- _c0: integer (nullable = true)
 |-- libro: string (nullable = true)
 |-- capitulo: integer (nullable = true)
 |-- verso: integer (nullable = true)
 |-- texto: string (nullable = true)
 |-- texto_limpio: string (nullable = true)
 |-- longitud_texto: integer (nullable = true)
 |-- tokens: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- num_palabras: integer (nullable = false)


[T4] Muestra del dataset transformado:
+-------+--------+-----+----------------------------------------------------------------------+--------------+------------+
|  libro|capitulo|verso|                                                          texto_limpio|longitud_texto|num_palabras|
+-------+--------+-----+----------------------------------------------------------------------+--------------+------------+
|Génesis|       1|    1| el relato de la creación en el principio dios creó los cielos y la...|            74|          16|


# CELDA 9 — LOAD (guardar en Parquet)

In [19]:
df_transformado.write \
    .mode("overwrite") \
    .partitionBy("libro") \
    .parquet(RUTA_PARQUET)

26/03/26 04:23:16 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , libro, capitulo, verso, texto
 Schema: _c0, libro, capitulo, verso, texto
Expected: _c0 but found: 
CSV file: file:///opt/notebooks/proyecto/data/biblia_ntv_.csv


In [20]:
print(f"[LOAD] Parquet guardado en: {RUTA_PARQUET}")


[LOAD] Parquet guardado en: /opt/notebooks/proyecto/data/biblia_ntv_procesada.parquet


In [21]:
# Verificación: releer y contar
df_verificacion = spark.read.parquet(RUTA_PARQUET)
print(f"[LOAD] Registros verificados en Parquet: {df_verificacion.count():,}")

[LOAD] Registros verificados en Parquet: 31,100


In [22]:
# Ver particiones generadas (carpetas por libro)
particiones = [
    d for d in os.listdir(RUTA_PARQUET)
    if d.startswith("libro=")
]
print(f"[LOAD] Particiones (libros) generadas: {len(particiones)}")
print(f"[LOAD] Primeras 10: {sorted(particiones)[:10]}")

[LOAD] Particiones (libros) generadas: 66
[LOAD] Primeras 10: ['libro=1 Corintios', 'libro=1 Crónicas', 'libro=1 Juan', 'libro=1 Pedro', 'libro=1 Reyes', 'libro=1 Samuel', 'libro=1 Tesalonicenses', 'libro=1 Timoteo', 'libro=2 Corintios', 'libro=2 Crónicas']


# CELDA 10 — ANÁLISIS DE PALABRAS CLAVE

In [23]:
# Leer desde Parquet (simula entorno de producción)
df = spark.read.parquet(RUTA_PARQUET)
total_ver = df.count()


In [24]:
def analizar_palabra(df, palabra):
    """
    Filtra versículos que contienen la palabra como token completo,
    calcula frecuencia total y agrupa por libro.

    Usa array_contains para evitar coincidencias parciales:
    "vida" NO coincide con "avidez" ni "evidencia".
    """
    df_filtrado = df.filter(
        array_contains(col("tokens"), palabra)
    )
    total = df_filtrado.count()
    pct   = (total / total_ver) * 100

    print(f"\n  {'─'*48}")
    print(f"  Palabra            : «{palabra}»")
    print(f"  Versículos hallados: {total:,}")
    print(f"  % del corpus       : {pct:.2f}%")
    print(f"  {'─'*48}")

    df_por_libro = df_filtrado \
        .groupBy("libro") \
        .agg(count("*").alias("frecuencia")) \
        .orderBy(desc("frecuencia"))

    return df_filtrado, df_por_libro

In [25]:
# ── «esperanza» ───────────────────────────────────────────────────────────────
df_esperanza, df_esp_por_libro = analizar_palabra(df, "esperanza")

print("\n=== TOP 15 LIBROS — «esperanza» ===")
df_esp_por_libro.show(15)

print("=== MUESTRA DE VERSÍCULOS — «esperanza» ===")
df_esperanza.select(
    "libro", "capitulo", "verso", "texto"
).show(5, truncate=100)


  ────────────────────────────────────────────────
  Palabra            : «esperanza»
  Versículos hallados: 146
  % del corpus       : 0.47%
  ────────────────────────────────────────────────

=== TOP 15 LIBROS — «esperanza» ===
+----------------+----------+
|           libro|frecuencia|
+----------------+----------+
|          Salmos|        31|
|        Jeremías|        18|
|             Job|        15|
|         Romanos|        10|
|          Hechos|         7|
|         Hebreos|         7|
|          Isaías|         6|
|      Proverbios|         6|
|        Ezequiel|         4|
|         1 Pedro|         4|
|         Miqueas|         3|
|         Efesios|         3|
|   Lamentaciones|         3|
|       1 Timoteo|         3|
|1 Tesalonicenses|         3|
+----------------+----------+
only showing top 15 rows

=== MUESTRA DE VERSÍCULOS — «esperanza» ===
+------+--------+-----+----------------------------------------------------------------------------------------------------+
| li

In [26]:
# ── «vida» ────────────────────────────────────────────────────────────────────
df_vida, df_vida_por_libro = analizar_palabra(df, "vida")

print("\n=== TOP 15 LIBROS — «vida» ===")
df_vida_por_libro.show(15)

print("=== MUESTRA DE VERSÍCULOS — «vida» ===")
df_vida.select(
    "libro", "capitulo", "verso", "texto"
).show(5, truncate=100)


  ────────────────────────────────────────────────
  Palabra            : «vida»
  Versículos hallados: 874
  % del corpus       : 2.81%
  ────────────────────────────────────────────────

=== TOP 15 LIBROS — «vida» ===
+------------+----------+
|       libro|frecuencia|
+------------+----------+
|      Salmos|        82|
|  Proverbios|        53|
|         Job|        48|
|Deuteronomio|        41|
|     Génesis|        39|
|        Juan|        38|
|     Romanos|        35|
| Eclesiastés|        33|
|      Isaías|        28|
|    1 Samuel|        28|
|     1 Reyes|        22|
|       Josué|        22|
| Apocalipsis|        22|
|    2 Samuel|        21|
|    Ezequiel|        20|
+------------+----------+
only showing top 15 rows

=== MUESTRA DE VERSÍCULOS — «vida» ===
+------+--------+-----+----------------------------------------------------------------------------------------------------+
| libro|capitulo|verso|                                                                        

# CELDA 11 — COMPARACIÓN CRUZADA

In [27]:
df_esp_ren  = df_esp_por_libro.withColumnRenamed("frecuencia", "frec_esperanza")
df_vida_ren = df_vida_por_libro.withColumnRenamed("frecuencia", "frec_vida")

df_comparacion = df_esp_ren.join(df_vida_ren, on="libro", how="outer") \
    .fillna(0, subset=["frec_esperanza", "frec_vida"]) \
    .withColumn(
        "predominante",
        when(col("frec_vida") > col("frec_esperanza"), "vida")
        .when(col("frec_esperanza") > col("frec_vida"), "esperanza")
        .otherwise("equilibrio")
    ) \
    .withColumn(
        "diferencia",
        F.abs(col("frec_vida") - col("frec_esperanza"))
    )

In [28]:
print("=== COMPARACIÓN POR LIBRO (orden por frec_vida) ===")
df_comparacion.orderBy(desc("frec_vida")).show(20)


=== COMPARACIÓN POR LIBRO (orden por frec_vida) ===
+------------+--------------+---------+------------+----------+
|       libro|frec_esperanza|frec_vida|predominante|diferencia|
+------------+--------------+---------+------------+----------+
|      Salmos|            31|       82|        vida|        51|
|  Proverbios|             6|       53|        vida|        47|
|         Job|            15|       48|        vida|        33|
|Deuteronomio|             0|       41|        vida|        41|
|     Génesis|             1|       39|        vida|        38|
|        Juan|             1|       38|        vida|        37|
|     Romanos|            10|       35|        vida|        25|
| Eclesiastés|             2|       33|        vida|        31|
|    1 Samuel|             1|       28|        vida|        27|
|      Isaías|             6|       28|        vida|        22|
|     1 Reyes|             0|       22|        vida|        22|
| Apocalipsis|             0|       22|        vida|

In [29]:
print("=== LIBROS DONDE PREDOMINA «esperanza» ===")
df_comparacion.filter(col("predominante") == "esperanza") \
    .orderBy(desc("frec_esperanza")).show(15)

=== LIBROS DONDE PREDOMINA «esperanza» ===


+-------+--------------+---------+------------+----------+
|  libro|frec_esperanza|frec_vida|predominante|diferencia|
+-------+--------------+---------+------------+----------+
|Miqueas|             3|        0|   esperanza|         3|
| Esdras|             1|        0|   esperanza|         1|
|  Oseas|             1|        0|   esperanza|         1|
+-------+--------------+---------+------------+----------+



In [30]:
print("=== LIBROS DONDE PREDOMINA «vida» ===")
df_comparacion.filter(col("predominante") == "vida") \
    .orderBy(desc("frec_vida")).show(15)

=== LIBROS DONDE PREDOMINA «vida» ===
+------------+--------------+---------+------------+----------+
|       libro|frec_esperanza|frec_vida|predominante|diferencia|
+------------+--------------+---------+------------+----------+
|      Salmos|            31|       82|        vida|        51|
|  Proverbios|             6|       53|        vida|        47|
|         Job|            15|       48|        vida|        33|
|Deuteronomio|             0|       41|        vida|        41|
|     Génesis|             1|       39|        vida|        38|
|        Juan|             1|       38|        vida|        37|
|     Romanos|            10|       35|        vida|        25|
| Eclesiastés|             2|       33|        vida|        31|
|    1 Samuel|             1|       28|        vida|        27|
|      Isaías|             6|       28|        vida|        22|
| Apocalipsis|             0|       22|        vida|        22|
|     1 Reyes|             0|       22|        vida|        22|
| 

In [31]:
# ── Versículos con AMBAS palabras ─────────────────────────────────────────────
df_ambas = df.filter(
    array_contains(col("tokens"), "esperanza") &
    array_contains(col("tokens"), "vida")
)
print(f"\n  Versículos con AMBAS palabras: {df_ambas.count():,}")
df_ambas.select("libro", "capitulo", "verso", "texto") \
    .show(5, truncate=100)


  Versículos con AMBAS palabras: 10


+-----------+--------+-----+----------------------------------------------------------------------------------------------------+
|      libro|capitulo|verso|                                                                                               texto|
+-----------+--------+-----+----------------------------------------------------------------------------------------------------+
|     Salmos|      62|   10|10 No te ganes la vida mediante la extorsión ni pongas tu esperanza en el robo. Y si tus riquezas...|
|        Job|       4|    6|              6 ¿No te da confianza tu reverencia a Dios? ¿No te da esperanza tu vida de integridad?|
|        Job|      27|    8|       8 Pues, ¿qué esperanza tienen los incrédulos cuando Dios acaba con ellos y les quita la vida?|
| Proverbios|      13|   12|           12 La esperanza postergada aflige al corazón, pero un sueño cumplido es un árbol de vida.|
|1 Corintios|      15|   19|19 Y si nuestra esperanza en Cristo es solo para esta vida, so

# CELDA 12 — RESUMEN FINAL

In [32]:
total_esp  = df_esperanza.count()
total_vida = df_vida.count()

In [33]:
print("\n" + "=" * 55)
print("  RESUMEN ESTADÍSTICO FINAL")
print("=" * 55)
print(f"  Total versículos en corpus     : {total_ver:,}")
print(f"  Versículos con «esperanza»     : {total_esp:,}  ({(total_esp/total_ver)*100:.2f}%)")
print(f"  Versículos con «vida»          : {total_vida:,}  ({(total_vida/total_ver)*100:.2f}%)")
print(f"  Ratio vida / esperanza         : {total_vida/total_esp:.2f}x")
print(f"  Versículos con ambas palabras  : {df_ambas.count():,}")
print("=" * 55)
print(f"\n  Spark UI disponible en → http://localhost:4040")
print(f"  Parquet guardado en    → {RUTA_PARQUET}")



  RESUMEN ESTADÍSTICO FINAL
  Total versículos en corpus     : 31,100
  Versículos con «esperanza»     : 146  (0.47%)
  Versículos con «vida»          : 874  (2.81%)
  Ratio vida / esperanza         : 5.99x
  Versículos con ambas palabras  : 10

  Spark UI disponible en → http://localhost:4040
  Parquet guardado en    → /opt/notebooks/proyecto/data/biblia_ntv_procesada.parquet
